# Gmail RAG Agent
Ask questions about your emails using local Ollama models (no API costs)

In [1]:
!pip install langchain-ollama langchain-chroma chromadb langchain

You should consider upgrading via the '/Users/talant/Desktop/RAG-Gmail/.venv/bin/python3 -m pip install --upgrade pip' command.


## Step 1: Pull the embedding model (run once)

In [2]:
import subprocess
result = subprocess.run(['ollama', 'pull', 'nomic-embed-text'], capture_output=True, text=True)
print(result.stdout or result.stderr)

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 



## Step 2: Load and clean emails

In [1]:
import json
import re

with open('emails.json', 'r') as f:
    emails = json.load(f)

def clean_body(text):
    # Remove HTML style/script blocks (if still present)
    text = re.sub(r'<style[^>]*>.*?</style>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<script[^>]*>.*?</script>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    # Remove remaining HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove raw CSS blocks: anything like "selector { ... }" left over after tag stripping
    text = re.sub(r'[^{}\n]*\{[^{}]*\}', ' ', text)
    # Remove @import and @media lines
    text = re.sub(r'@\w+[^;\n]*[;\n]', ' ', text)
    # Decode common HTML entities
    text = text.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&#39;', "'").replace('&quot;', '"')
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Format each email as a clean document
documents = []
for email in emails:
    body = clean_body(email.get('body', ''))
    if not body:
        continue
    doc = f"""Subject: {email['subject']}
From: {email['from']}
Date: {email['date']}
Body: {body[:1500]}"""
    documents.append({
        'text': doc,
        'metadata': {
            'id': email['id'],
            'subject': email['subject'],
            'from': email['from'],
            'date': email['date']
        }
    })

print(f"Prepared {len(documents)} emails for indexing")

# Preview the REQ344466 email to verify it's clean
for d in documents:
    if 'REQ344466' in d['metadata']['subject']:
        print(f"\n--- {d['metadata']['subject']} ---")
        print(d['text'][:600])

Prepared 84 emails for indexing

--- REQ344466 Business Analysis Manager - Care Demand Planning Insights and Automation ---
Subject: REQ344466 Business Analysis Manager - Care Demand Planning Insights and Automation
From: T-Mobile Careers <tmobile@myworkday.com>
Date: Fri, 27 Mar 2026 03:19:27 +0000
Body: @media only screen and (max-width:600px){ } Hi Talant , Thank you for your interest in the Business Analysis Manager – Care Demand Planning Insights and Automation role. We truly appreciate you taking the time to apply. We’ve received a strong level of interest for this opportunity, and the team is still in the process of reviewing applications. As a result, there have been some delays in moving forward. We apprec

--- Thank You for Your Interest – REQ344466 Business Analysis Manager - Care Demand Planning Insights and Automation ---
Subject: Thank You for Your Interest – REQ344466 Business Analysis Manager - Care Demand Planning Insights and Automation
From: T-Mobile Careers <tmobile

## Step 3: Embed and store in ChromaDB

In [2]:
import shutil
import os
import chromadb
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.schema import Document

embeddings = OllamaEmbeddings(model='nomic-embed-text')

lc_docs = [
    Document(page_content=d['text'], metadata=d['metadata'])
    for d in documents
]

# Use Gmail message IDs so re-running this cell doesn't create duplicates
doc_ids = [d['metadata']['id'] for d in documents]

CHROMA_PATH = '/Users/talant/Desktop/RAG-Gmail/chroma_db'

# Clear old DB completely
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

# Build vectorstore fresh in one shot (avoids open-file conflicts)
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
    collection_name='gmail_emails',
    ids=doc_ids,
)

count = vectorstore._collection.count()
print(f"Indexed {count} emails into ChromaDB at {CHROMA_PATH}")
print(f"Expected: {len(lc_docs)} — {'OK' if count == len(lc_docs) else 'DUPLICATES DETECTED'}")

/Users/talant/Desktop/RAG-Gmail/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Indexed 84 emails into ChromaDB at /Users/talant/Desktop/RAG-Gmail/chroma_db
Expected: 84 — OK


## Step 4: Build the RAG chain

In [3]:
from langchain_ollama import ChatOllama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

llm = ChatOllama(model='llama3.1')

retriever = vectorstore.as_retriever(search_kwargs={'k': 10})

prompt_template = """You are a helpful assistant that answers questions about the user's Gmail inbox.
Use the email context below to answer the question. If you can't find the answer, say so.

Email context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=['context', 'question']
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    chain_type_kwargs={'prompt': prompt},
    return_source_documents=True
)

print("RAG chain ready!")

RAG chain ready!


## Step 5: Ask questions about your emails

In [4]:
def ask(question):
    result = qa_chain.invoke({'query': question})
    print(f"Q: {question}")
    print(f"\nA: {result['result']}")
    print("\n--- Retrieved emails ---")
    for doc in result['source_documents']:
        print(f" - {doc.metadata['subject']} | {doc.metadata['from']}")
    print()
    
# Try some questions!
ask("Did i receive any answer from t_mobile careers  regarding my job application REQ344466?")

Q: Did i receive any answer from t_mobile careers  regarding my job application REQ344466?

A: Yes, you did receive an answer from T-Mobile Careers regarding your job application for the Business Analysis Manager - Care Demand Planning Insights and Automation role (REQ344466). 

You received two emails from them:

1. A message on March 6, 2026, saying that they appreciate your interest in the opportunity but are taking thoughtful care to review each candidate thoroughly.
2. Another message on March 27, 2026, informing you that there have been some delays due to a strong level of interest for this opportunity and that they will be in touch with updates as soon as possible.

These emails indicate that your application is still being considered, but the timeline for moving forward has been delayed due to high demand.

--- Retrieved emails ---
 - RE: Clarification on Start Date and Assignment Details | "Del Angel Gamboa, Jaime" <jaime.delangelgamboa@t-mobile.com>
 - Availability for interv